# 00 — Build the RAG dataset (retrieval corpus)

Thin wrapper around **`sample_generation.get_rag_corpus()`**. The corpus build now lives in the centralized `sample_generation.py` util, alongside the other batches (`get_tuning_sample` / `get_robustness_batch` / `get_test_batch`) — not in `rag_utils`.

It creates **`data/processed/rag_corpus.csv`** — the pool of labelled *precedent* loans the 05_rag notebooks retrieve from: the full 2012–2014 frame with every evaluation batch removed, so no eval (or test) loan can ever be retrieved.

- **Full corpus** when the raw `data/raw/accepted_2007_to_2018Q4.csv.gz` is present.
- **Dev fallback** (~100-row `tuning_sample`, disjoint from `robustness_batch`) when it is absent — lets Phase 5 run before the raw file is added.

The `robustness_batch` (eval set) and held-out `test_batch` are always excluded, and zero overlap with the eval set is asserted on every build.

You don't strictly need this notebook: `python sample_generation.py` builds every batch including the corpus, and 05a/b/c call `get_rag_corpus()` lazily (load-if-exists). It's here as the documented Phase-5 data step.

In [1]:
import sys; sys.path.insert(0, '..')
from sample_generation import get_rag_corpus, get_robustness_batch
import rag_utils as R   # assert_no_leakage

In [2]:
# force=True regenerates. With the raw .csv.gz present this builds the full large
# corpus; without it, the ~100-row tuning_sample dev fallback. Plain get_rag_corpus()
# (no force) is load-if-exists and returns the committed file.
corpus = get_rag_corpus(force=True)

# Leakage guard: the corpus must be disjoint from the evaluation set (robustness_batch).
robustness = get_robustness_batch()
R.assert_no_leakage(corpus, robustness)
print(f'RAG corpus rows       : {len(corpus)}')
print(f'Robustness (eval) rows: {len(robustness)}')
print('Leakage check         : PASSED (corpus ∩ robustness = ∅)')

Generated rag_corpus.csv: 260579 rows -> /Users/alemz/Projects/Github/Sabadell_Capstone/data/processed/rag_corpus.csv
RAG corpus rows       : 260579
Robustness (eval) rows: 100
Leakage check         : PASSED (corpus ∩ robustness = ∅)


In [3]:
# Quick profile of the corpus
co = int((corpus['loan_status'] == 0).sum())
print(f'Charged Off: {co}  |  Fully Paid: {len(corpus) - co}')
corpus.head()

Charged Off: 46270  |  Fully Paid: 214309


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,mths_since_last_delinq,acc_open_past_24mths,credit_history_years,has_past_delinq,total_pymnt,total_rec_prncp,total_rec_int,recoveries,collection_recovery_fee,funded_amnt
0,10400.0,36 months,6.99,321.08,A,A3,8.0,MORTGAGE,58000.0,Not Verified,...,42.0,7.0,25.248460,1,6611.69,5217.75,872.67,521.27,93.8286,10400.0
1,15000.0,60 months,12.39,336.64,C,C1,10.0,RENT,78000.0,Source Verified,...,999.0,5.0,20.334018,0,17392.37,15000.00,2392.37,0.00,0.0000,15000.0
2,9600.0,36 months,13.66,326.53,C,C3,10.0,RENT,69000.0,Source Verified,...,999.0,8.0,22.080767,0,9973.43,9600.00,373.43,0.00,0.0000,9600.0
3,7650.0,36 months,13.66,260.20,C,C3,0.0,RENT,50000.0,Source Verified,...,999.0,6.0,12.334018,0,2281.98,704.38,339.61,1237.99,222.8382,7650.0
4,21425.0,60 months,15.59,516.36,D,D1,6.0,RENT,63800.0,Source Verified,...,60.0,4.0,11.334702,1,25512.20,21425.00,4087.20,0.00,0.0000,21425.0
